# sklearn : cross-validation
* 학습 시 데이터를 train, test 두 종류로만 나누어 테스트-학습을 반복하면 hyper-parameter의 튜닝 편향성이 생겨 Overfitting의 위험이 존재한다. 이 문제를 방지하기 위해서 교차 검증을 이용한다.
* 교차 검증은 데이터 편중을 막고, hyper-parameter튜닝 등 최종 평가 이전에 학습된 모델을 다양하게 평가하기 위한 방법

1. K-Fold CV(Cross Validation)
	* 학습 데이터 세트를 K개의 폴드 데이터 세트로 분리한 뒤, K번에 걸쳐 K-1개의 폴드 세트를 학습용 데이터 세트로, 1개의 폴드를 검증 데이터 세트로 설정한 뒤, 각 폴드 세트에 대해 학습 및 평가를 수행한다. 이 과정을 K번 반복한 뒤에 모든 평가 지표에 대한 평균을 통해 최종 평가 지표 값을 얻는 방식이다.
	* `sklearn.model_selection.KFold`를 이용해 객체를 생성하고, `KFold.split(features)` 메서드를 이용해서, train fold dataset와 validation fold dataset의 '인덱스'를 반환받는다.
		* 실제 데이터의 인덱싱은 개발자가 직접 명시해야한다.

2. Stratified K Fold
	* 원본 데이터의 레이블 분포를 고려한 뒤, 이 분포와 동일하게 학습과 검증 데이터 세트를 분배한다.
	* 불균형한 분포도를 가진 레이블 데이터 집합을 위한 K Fold 방식
	* `sklearn.model_selection.StratifiedKFold`를 이용해 객체를 생성하고, `StratifiedKFold.split(features, label)` 메서드를 이용해서, train fold dataset와 validation fold dataset의 '인덱스'를 반환받는다.
		* 실제 데이터의 인덱싱은 개발자가 직접 명시해야한다.
	* 불균형한 레이블 데이터 세트에 대해서는 반드시 StratifiedKFold를 이용하자.
	* 회귀에서는 해당 방식을 사용하지 않는다.

3. `cross_val_score()`
	* 하나의 성능 평가 지표(Ex, accuracy 등)에 대해서 k fold(+ stratified k fold) 각각의 평가 결괏값을 반환해주는 API
	* 한번에 하나의 성능 평가 지표만 판단 가능
	* 일반적으로 해당 API의 출력 결과를 평균내어 최종 평가 지표로 삼음

4. `cross_validate()`
	* 여러 성능 지표를 한번에 평가하고 싶을 때 사용한다.
	* 추가로, 수행 시간도 측정 가능하다.

5. `GridSearchCV()`
	* 다양한 HyperParameter 조합을 이용해 성능을 평가하고, 최적의 파라미터를 도출해주는 API
	* 교차 검증을 기반으로 하이퍼 파라미터의 최적값을 찾아준다. 그 대신, 시간이 오래 걸린다.

### sklearn 교차 검증 practice : kfold
- 단, kfold를 이용 시, label의 분포를 고르게 만들지 않는다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import numpy as np

iris = load_iris()
features = iris.data
label = iris.target

df_clf = DecisionTreeClassifier(random_state=156)

kfold = KFold(n_splits=5)
cv_accuracy = []

for train_index, test_index in kfold.split(features) :
	X_train, X_test = features[train_index], features[test_index]
	y_train, y_test = label[train_index], label[test_index]

	df_clf.fit(X_train, y_train)
	pred = df_clf.predict(X_test)
	accuracy = np.round(accuracy_score(y_test, pred), 4) # 소숫점 이하 5자리에서 반올림

	cv_accuracy.append(accuracy)

print(cv_accuracy)
print(np.mean(cv_accuracy))


- 아래와 같은 경우 KFold이용 시 정확도는 0이 된다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold

import numpy as np

iris = load_iris()
features = iris.data
label = iris.target

df_clf = DecisionTreeClassifier(random_state=156)

kfold = KFold(n_splits=3)
cv_accuracy = []


for train_index, test_index in kfold.split(features) :
	X_train, X_test = features[train_index], features[test_index]
	y_train, y_test = label[train_index], label[test_index]

	df_clf.fit(X_train, y_train)
	pred = df_clf.predict(X_test)
	accuracy = np.round(accuracy_score(y_test, pred), 4) # 소숫점 이하 5자리에서 반올림

	cv_accuracy.append(accuracy)

print(cv_accuracy)
print(np.mean(cv_accuracy))


### sklearn 교차 검증 practice : StratifiedKFold
- label dataset의 분포를 확인해서 train/validation 데이터가 최대한 고르게 분포되도록 나눈다.
- 일반적으로 분류는 Stratified K fold를 이용한다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold

import numpy as np

iris = load_iris()
features = iris.data
labels = iris.target

df_clf = DecisionTreeClassifier(random_state=156)
skf = StratifiedKFold(n_splits=3)

cv_accuracy = []

for train_idx, test_idx in skf.split(features, labels) :
	X_train, X_test = features[train_idx], features[test_idx]
	y_train, y_test = labels[train_idx], labels[test_idx]

	df_clf.fit(X_train, y_train)
	pred = df_clf.predict(X_test)

	accuracy = np.round(accuracy_score(y_test, pred), 4)

	print(accuracy)
	cv_accuracy.append(accuracy)

print (np.mean(cv_accuracy))

### sklearn 교차 검증 practice : cross_val_score()
- KFold객체를 생성하고, split()을 통해 얻은 인덱스로 인덱싱을 수행해서 학습 / 검증 데이터 셋을 분리하고, estimator를 학습 및 예측하는 과정을 한 번에 수행할 수 있게 해준다.
- 추가로, cross_validate() 이용 시 여러 개의 평가 지표를 한번에 체크하고, 수행 시간도 제공받을 수 있다.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, cross_validate
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score

iris = load_iris()
datas = iris.data
labels = iris.target

dt_clf = DecisionTreeClassifier(random_state=156)

accuracys = cross_val_score(estimator=dt_clf, X=datas, y=labels, scoring='accuracy', cv=3)

print(np.round(accuracys, 4))
print(np.round(np.mean(accuracys), 4))

### practice : GridSearchCV
- 한번에 교차 검증 및 최적 하이퍼 파라미터 튜닝까지 수행하는 방법
- 주어진 하이퍼 파라미터 조합에 대한 검증을 모두 수행하는 대신, 시간이 엄청 오래 걸린다.
- `GridSearchCV`객체는 다양한 인스턴스 변수를 가진다.
	1. `best_params_` : 최적의 파라미터 조합을 알려준다.
	2. `best_scores_` : 최적의 파라미터 조합을 적용했을 때의 최고 정확도를 알려준다.
	3. `best_estimator_` : `refit=True`일 때, 최적의 파라미터를 이용해 estimator를 학습시킨 상태의 객체를 참조하고 있는 변수.
		- 이 변수를 받아와서 예측(predict)를 수행한다.

In [ ]:
from sklearn.datasets import load_iris
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import accuracy_score

import pandas as pd

iris = load_iris()
datas = iris.data
labels = iris.target

dt_clf = DecisionTreeClassifier(random_state=156)
X_train, X_test, y_train, y_test = train_test_split(datas, labels, test_size=0.2, random_state=121)

parameters={
	'max_depth':[1, 2, 3],
	'min_samples_split' :[2, 3]
}

grid_dtree = GridSearchCV(dt_clf, param_grid=parameters, cv=3, refit=True)
grid_dtree.fit(X_train, y_train)

#scores_df = pd.DataFrame(grid_dtree.cv_results_)
#scores_df[['params', 'mean_test_score', 'rank_test_score', 'split0_test_score', 'split1_test_score', 'split2_test_score']]

best_estimator = grid_dtree.best_estimator_
pred = best_estimator.predict(X_test)
print(accuracy_score(y_test, pred))